### Data Prep

## **Dynamic Parametarization (changing folders)**

In [ ]:
# Environment configuration — replace the placeholder in the Databricks widget.
dbutils.widgets.text("storage_account", "<YOUR_STORAGE_ACCOUNT>")
storage_account = dbutils.widgets.get("storage_account")
if not storage_account or storage_account.startswith("<"):
    raise ValueError("Set the storage_account widget before running.")


In [ ]:
dbutils.widgets.text("file_name", "")

In [ ]:
p_file_name = dbutils.widgets.get("file_name")

### **PySpark Data Structured Streaming  (Reading)**

#### Databricks Autoloader setting up which reads streaming data from Azure Data Lake (ADLS Gen2) in Parquet format.

In [ ]:
df = spark.readStream.format("cloudFiles")\
    .option("cloudFiles.format", "parquet")\
    .option("cloudFiles.schemaLocation", f"abfss://bronze@{storage_account}.dfs.core.windows.net/checkpoint_{p_file_name}")\
    .load(f"abfss://source@{storage_account}.dfs.core.windows.net/{p_file_name}")

### **Data Writing (only Incremental Data)** 

In [ ]:
df.writeStream.format("parquet")\
    .outputMode("append")\
    .option("checkpointLocation", f"abfss://bronze@{storage_account}.dfs.core.windows.net/checkpoint_{p_file_name}")\
    .option("path", f"abfss://bronze@{storage_account}.dfs.core.windows.net/{p_file_name}")\
    .trigger(once=True)\
    .start()

In [ ]:
df = spark.read.format("parquet")\
    .load(f"abfss://bronze@{storage_account}.dfs.core.windows.net/{p_file_name}")
display(df)